# ControlNet Conditioning

Text alone is a clumsy control signal. ControlNet lets you clone a pretrained diffusion model and steer it with a depth map, pose skeleton, scribble, or edge image. LoRA lets you fine-tune a 2B-parameter model by training 10 million parameters. Together they turned Stable Diffusion from a toy into 2026 image pipeline that ships at every agency.

## Problem Definition

A prompt like "a woman in a red dress walking a dog on a busy street" gives the model no information about where the dog is, what pose the woman is in, or the perspective of the street. Text pins down about 10% of what you need to specify an image. The rest is visual and cannot be described efficiently in words.

Training a new conditional model from scratch for every signal (pose, depth, canny, segmentation) is prohibitive. **Keep backbone frozen, attach a small side-network that reads the conditioning**, and have it nudge the backbone's intermediate features.

You also want to teach the model new concepts (your face, your product, your style) without retraining the full model. You want a 100x smaller data. That is LoRA -- low rank adapters that plug into existing attention weights.

ControlNet + LoRA + text = the 2026 pratitioner's toolkit.

## Basic Concept

ControlNet clones the encoder; LoRA adds low-rank deltas.

### ControlNet

Take a pretrained SD, clone the encoder half of the U-Net, Freeze the original. Train the clone to accept an extra conditioning input (edges, depth, pos). Connect the clone back to the decoder half of the original with zero-convolution skip connections (1x1 convs initialized to zero -- start as a no-op, learn a delta).

```
SD U-Net deocer:  ... <-- orig_enc_features + zero_conv(controlnet_enc(condition))
```

Zero-conv init means ControlNet starts as identify, no harm even before tranining. Trainig on 1M **(prompt, condition, image) triples** with standard diffusion loss.

Per-modality ControlNets ship as small side models. You can compose them at inference.
```
features += weight_a * control_a(depth) + weight_b * control_b(pose)
```

### IP-Adapter

A tiny adapter that accepts an image as conditioning. Uses the CLIP image encoder to produce image tokens, injects them into cross-attention alongside text tokens.

# Build your Own

In [ ]:
import torch
import torch.nn as nn
import random

torch.manual_seed(42)
random.seed(42)

w_side = nn.Parameter(torch.Tensor([random.gauss(0, 0.1)]))
gate = nn.Parameter(torch.Tensor([0.0]))
lr = 0.03
trace = []

optim = torch.optim.SGD([w_side, gate], lr=lr)

for step in range(1000):
    # base: f_base(x) = x (frozen)
    # side: f_side(x, c) = c (learnable weight w_side)
    # gated: out = f_base + gate * w_side * c
    x = torch.tensor(random.gauss(0, 1))
    c = torch.tensor(random.choice([-1.0, 1.0]))
    target = x + 0.7 * c

    perd = x + gate * w_side * c
    optim.zero_grad()
    loss = (perd - target) ** 2
    loss.backward()
    optim.step()

    if (step + 1) % 100 == 0:
        print(f"Step {step + 1}: Gate={gate.item():.4f}, w_side={w_side.item():.4f}, loss={loss.item():.4f}")
        print(f"Gate * w_side = {gate.item() * w_side.item():.4f}")

Step 100: Gate=-0.3923, w_side=-0.3925, loss=0.3089
Step 200: Gate=-0.8364, w_side=-0.8365, loss=0.0000
Step 300: Gate=-0.8366, w_side=-0.8367, loss=0.0000
Step 400: Gate=-0.8366, w_side=-0.8367, loss=0.0000
Step 500: Gate=-0.8366, w_side=-0.8367, loss=0.0000
Step 600: Gate=-0.8366, w_side=-0.8367, loss=0.0000
Step 700: Gate=-0.8366, w_side=-0.8367, loss=0.0000
Step 800: Gate=-0.8366, w_side=-0.8367, loss=0.0000
Step 900: Gate=-0.8366, w_side=-0.8367, loss=0.0000
Step 1000: Gate=-0.8366, w_side=-0.8367, loss=0.0000
